# 5. Control de un robot aéreo en 3D

Se trabajará con el modelo no lineal de un robot aéreo (quadcópter) en 3 dimensiones. 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from Robot3D import (RobotAereo3D, Rrpy, convert_control_to_motors,
                     AnimacionRobot3D, plot_trajectory)

## Controlador usando linealización por realimentación

Se utiliza un esquema de "feedback linearization" (linealización por realimentación) para cancelar los términos no lineales del modelo e insertar un comportamiento deseado al sistema. Debido a la subactuación, se emplea un control de tipo cascada.

In [ ]:
class Controlador:

    def __init__(self, Ix, Iy, Iz,
                 kp_z=1.0, kd_z=1.0, kp_x=1.0, kd_x=1.0,
                 kp_y=1.0, kd_y=1.0, kp_roll=1.0, kp_pitch=1.0, kp_yaw=1.0,
                 kp_p=1.0, kp_q=1.0, kp_r=1.0):

        self.Ix = Ix; self.Iy = Iy; self.Iz = Iz
        self.kp_z = kp_z; self.kd_z = kd_z; self.kp_x = kp_x
        self.kd_x = kd_x; self.kp_y = kp_y; self.kd_y = kd_y
        self.kp_roll = kp_roll; self.kp_pitch = kp_pitch; self.kp_yaw = kp_yaw
        self.kp_p = kp_p; self.kp_q = kp_q; self.kp_r = kp_r
        self.g= 9.81

    def control_altura(self, z_des, z, dz_des, dz, ddz_des, r33):
        """
        Control PD de la altitud (z). Genera u1
        """
        vz = ddz_des - self.kd_z*(dz-dz_des) - self.kp_z*(z-z_des) 
        u1 = (vz-self.g)/r33
        return u1

    def control_lateral(self, x_des, x, dx_des, dx, ddx_des,
                        y_des, y, dy_des, dy, ddy_des):
        """
        Control PD lateral (x,y)
        """
        vx = ddx_des - self.kd_x*(dx-dx_des) - self.kp_x*(x-x_des) 
        vy = ddy_des - self.kd_y*(dy-dy_des) - self.kp_y*(y-y_des) 
        return vx, vy

    def control_orientacion(self, vx, vy, psi_des, psi, p, q, r, R, u1):
        """
        Control de la orientación
        """
        # Velocidades de referencia
        p_d, q_d = self.control_roll_pitch(vx, vy, R, u1)
        r_d = self.control_yaw(psi_des, psi)
        # Control de la velocidad angular del cuerpo
        ux, uy, uz = self.body_rate_controller(p_d, q_d, r_d, p, q, r)
        return ux, uy, uz

    def control_roll_pitch(self, vx, vy, R, u1):
        """
        Control P de Roll-Pitch. Retorna velocidad angular x, y en sistema del robot (p, q)
        """
        r13_c = vx/u1
        r23_c = vy/u1
        v_13 = -self.kp_roll * (R[0,2]-r13_c)     # kp_roll = kp_phi
        v_23 = -self.kp_pitch * (R[1,2]-r23_c)    # kp_pitch = kp_theta

        Rmat = np.array([[R[1,0], -R[0,0]],
                         [R[1,1], -R[0,1]]])
        pq_d = 1/R[2,2] * Rmat @ np.array([v_13, v_23])
        p_d = pq_d[0]
        q_d = pq_d[1]
        return p_d, q_d

    def control_yaw(self, psi_des, psi):
        """
        Control P del yaw. Retorna velocidad angular z en sistema del robot (r)
        """
        v_psi = -self.kp_yaw * (psi-psi_des) 
        return v_psi

    def body_rate_controller(self, p_d, q_d, r_d, p, q, r):
        vp = -self.kp_p * (p-p_d)
        vq = -self.kp_q * (q-q_d)
        vr = -self.kp_r * (r-r_d)

        ux = vp * self.Ix + (self.Iz-self.Iy)*q*r
        uy = vq * self.Iy + (self.Ix-self.Iz)*p*r
        uz = vr * self.Iz + (self.Iy-self.Ix)*p*q

        return ux, uy, uz

### Generador de una trayectoria

Se utilizará una trayectoria que describe una forma de 8 en tres dimensiones que mantiene al robot siempre orientado a lo largo de la dirección del movimiento. La trayectoria es:

$$
\begin{align}
x &= \sin{\omega_x t} \\
y &= \cos{\omega_y t} \\
z &= \cos{\omega_z t} \\
\end{align}
$$
con $\omega_z = \omega_y = \omega_x/2$.

In [ ]:
def trayectoria(tiempo_total, omega_x=0.8, omega_y=0.4, omega_z=0.4,
                ax=1.0, ay=1.0, az=1.0, dt=0.01):

    t = np.linspace(0.0, tiempo_total, int(tiempo_total/dt))
    x   = ax*np.sin(omega_x*t)
    dx  = ax*omega_x*np.cos(omega_x*t)
    ddx = -ax*omega_x**2*np.sin(omega_x*t)

    y   = ay*np.cos(omega_y*t)
    dy  = -ay*omega_y*np.sin(omega_y*t)
    ddy = -ay*omega_y**2*np.cos(omega_y*t)

    z   = az*np.cos(omega_z*t)
    dz  = -az*omega_z*np.sin(omega_z*t)
    ddz = - az*omega_z**2*np.cos(omega_z*t)

    psi = np.arctan2(dy, dx)

    return (x, dx, ddx), (y, dy, ddy), (z, dz, ddz), psi, t

### Trayectoria deseada

In [ ]:
# Parámetros de la trayectoria
tiempo_total = 20.0
dt = 0.01

# Generación de la trayectoria
xx, yy, zz, psi_des, t = trayectoria(tiempo_total, dt=dt)
x_des, dx_des, ddx_des = xx
y_des, dy_des, ddy_des = yy
z_des, dz_des, ddz_des = zz

# Forma de la trayectoria deseada
fig = plt.figure()
ax = plt.axes(projection='3d')
ax.plot(x_des, y_des, z_des)
plt.title('Trayectoria cartesiana deseada').set_fontsize(10)
ax.set_xlabel('$x$ [$m$]').set_fontsize(10)
ax.set_ylabel('$y$ [$m$]').set_fontsize(10)
ax.set_zlabel('$z$ [$m$]').set_fontsize(10)
plt.show();

## Seguimiento de trayectoria

In [ ]:
# Robot simulado
robot = RobotAereo3D()

# Estado inicial del robot con altura 0 y velocidad 0
X0 = np.array([x_des[0], y_des[0], z_des[0], 0.0, 0.0, psi_des[0],
               dx_des[0], dy_des[0], dz_des[0], 0.0, 0.0, 0.0])
robot.set_state(X0)

# Parámetros "estimados" del robot
Ix = 0.1; Iy = 0.1; Iz = 0.2; m = 0.5
km = 1.0; kf = 1.0
L = 0.566 / (2*np.sqrt(2))

# Controlador
controlador = Controlador(Ix, Iy, Iz,
                          kp_z=2.0, kd_z=1.0, kp_x=6.0, kd_x=4.0, kp_y=6.0, kd_y=4.0,
                          kp_roll=8.0, kp_pitch=8.0, kp_yaw=8.0,
                          kp_p=20.0, kp_q=20.0, kp_r=20.0)

In [ ]:
# Establecer el estado inicial
robot.set_state(X0)

# Factor para controlar más rápido el lazo interno
factor_lazo_interno = 10

# Almacenamiento del historial de los estados y control
Xt = np.zeros((t.size, 12))
Xt[0,:] = robot.X
Ut = np.zeros((factor_lazo_interno*(t.size-1), 4))

# Ejecución del vuelo
for i in range(0, t.size-1):

    # Estado actual del robot
    x = robot.X[0];     dx = robot.X[6]
    y = robot.X[1];     dy = robot.X[7]
    z = robot.X[2];     dz = robot.X[8]
    phi = robot.X[3]
    theta = robot.X[4]
    psi = robot.X[5]

    rpy = Rrpy(phi, theta, psi)
    # Controlador
    u1 = controlador.control_altura(z_des[i], z, dz_des[i], dz, ddz_des[i], rpy[2,2])
    ddx_c, ddy_c = controlador.control_lateral(x_des[i], x, dx_des[i], dx, ddx_des[i],
                                               y_des[i], y, dy_des[i], dy, ddy_des[i])

    for j in range(factor_lazo_interno):

        phi = robot.X[3]; theta = robot.X[4]; psi = robot.X[5]
        p = robot.X[9]; q = robot.X[10]; r = robot.X[11]

        rpy = Rrpy(phi, theta, psi)
        ux, uy, uz = controlador.control_orientacion(ddx_c, ddy_c, psi_des[i], psi,
                                                     p, q, r, rpy, u1)

        # Obtener la velocidad de los motores del controlador
        w1, w2, w3, w4 = convert_control_to_motors(u1, ux, uy, uz, m, L, kf, km)

        # Simulación
        robot.set_control(w1, w2, w3, w4)
        robot_state = robot.simulate(dt/factor_lazo_interno)

        # Historial de la señal de control
        Ut[i*factor_lazo_interno+j,0] = w1
        Ut[i*factor_lazo_interno+j,1] = w2
        Ut[i*factor_lazo_interno+j,2] = w3
        Ut[i*factor_lazo_interno+j,3] = w4

    # Generación del historial de las posiciones
    Xt[i+1,:] = robot_state

# Recuperación del estado
traj_x = Xt[:,0]; traj_y = Xt[:,1]; traj_z = Xt[:,2]
traj_phi = Xt[:,3]; traj_theta = Xt[:,4]; traj_psi = Xt[:,5]
traj_dx = Xt[:,6]; traj_dy = Xt[:,7]; traj_dz = Xt[:,8]

# Recuperación del control
traj_w1 = Ut[:,0]; traj_w2 = Ut[:,1]; traj_w3 = Ut[:,2]; traj_w4 = Ut[:,3]

In [ ]:
fig = plt.figure()
ax = plt.axes(projection='3d')

ax.plot(x_des, y_des, z_des, linestyle='-', marker='.', color='red')
ax.plot(traj_x, traj_y, traj_z, linestyle='-', color='blue')

plt.title('Trayectoria del robot').set_fontsize(10)
ax.set_xlabel('$x$ [$m$]').set_fontsize(10)
ax.set_ylabel('$y$ [$m$]').set_fontsize(10)
ax.set_zlabel('$z$ [$m$]').set_fontsize(10)
plt.legend(['Deseado','Ejecutado'],fontsize = 10)

ax.set_xlim(-1, 1)
ax.set_ylim(-1, 1)
ax.set_zlim(-1, 1)

plt.show()

In [ ]:
fig = plot_trajectory(traj_x, traj_y, traj_z)
fig.show()

In [ ]:
# Mostrar una animación del vuelo del robot
ts = 10              # Factor de muestreo
porcentaje = 0.8     # Porcentaje (0 a 1) del tiempo total
tf = int(np.floor(porcentaje*(tiempo_total/dt)))

traj = {'time': t[0:tf:ts],
        'position': np.array([traj_x[0:tf:ts], traj_y[0:tf:ts], traj_z[0:tf:ts]]).T,
        'orientation': np.array([traj_phi[0:tf:ts], traj_theta[0:tf:ts], traj_psi[0:tf:ts]]).T}

sim = AnimacionRobot3D()
sim.run_simulation(trajectory_data = traj)

In [ ]:
plt.figure(figsize=(12,3))

plt.subplot(1,2,1)
plt.plot(t, traj_x); plt.plot(t, x_des)
plt.title('Trayectoria en x')
plt.xlabel('$t$ [s]'); plt.ylabel('$x$ [$m$]')
plt.gca().invert_yaxis()
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.subplot(1,2,2)
plt.plot(t, traj_dx); plt.plot(t,dx_des)
plt.title('Velocidad en x (lateral)')
plt.xlabel('$t$ [s]'); plt.ylabel(r'$\dot x$ [$m/s$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.figure(figsize=(12,3))

plt.subplot(1,2,1)
plt.plot(t, traj_y); plt.plot(t,y_des)
plt.title('Trayectoria en y')
plt.xlabel('$t$ [s]'); plt.ylabel(r'$y$ [$m$]')
plt.gca().invert_yaxis()
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.subplot(1,2,2)
plt.plot(t, traj_dy); plt.plot(t,dy_des)
plt.title('Velocidad en y')
plt.xlabel('$t$ [s]'); plt.ylabel(r'$\dot y$ [$m/s$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.figure(figsize=(12,3))

plt.subplot(1,2,1)
plt.plot(t, traj_z); plt.plot(t,z_des)
plt.title('Trayectoria en z')
plt.xlabel('$t$ [s]'); plt.ylabel(r'$z$ [$m$]')
plt.gca().invert_yaxis()
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.subplot(1,2,2)
plt.plot(t, traj_dz); plt.plot(t,dz_des)
plt.title('Velocidad en z')
plt.xlabel('$t$ [s]'); plt.ylabel(r'$\dot z$ [$m/s$]')
plt.legend(['Ejecutado', 'Deseado'],fontsize = 10)

plt.show()

In [ ]:
err = np.sqrt((x_des-traj_x)**2 + (y_des-traj_y)**2 + (y_des-traj_z)**2)

plt.figure(figsize=(6,4))
plt.plot(t, err)
plt.title('Error en posición (X, Y, Z)').set_fontsize(10)
plt.xlabel('$t$ [$s$]').set_fontsize(10)
plt.ylabel('$e$ [$m$]').set_fontsize(10)
plt.show()

In [ ]:
plt.figure(figsize=(8,4))
tw = np.linspace(t[0], t[-1], len(traj_w1))
tskip = 10

plt.plot(tw[:-2:tskip], traj_w1[:-2:tskip], color='blue')
plt.plot(tw[:-2:tskip], traj_w2[:-2:tskip], color='red')
plt.plot(tw[:-2:tskip], traj_w3[:-2:tskip], color='green')
plt.plot(tw[:-2:tskip], traj_w4[:-2:tskip], color='black')

plt.title('Velocidades angulares de los motores').set_fontsize(15)
plt.xlabel('$t$ [$s$]').set_fontsize(10)
plt.ylabel(r'$\omega$ [$rad/s$]').set_fontsize(10)
plt.legend(['P 1','P 2','P 3','P 4' ],fontsize = 10)

plt.grid()
plt.show()

In [ ]:
plt.figure(figsize=(6,4))
plt.plot(t, psi_des, marker='.')
plt.plot(t, traj_psi)
plt.title('Ángulo Yaw').set_fontsize(10)
plt.xlabel('$t$ [$s$]').set_fontsize(10)
plt.ylabel(r'$\psi$ [$rad$]').set_fontsize(10)
plt.legend(['Deseado', 'Ejecutado'], fontsize = 10)
plt.show()